In [1]:
# Block 1: Imports
import os
import cv2
import numpy as np
import pandas as pd
import ast
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from collections import deque, Counter
import logging
import pkg_resources
# Keep Ultralytics quiet in notebook output
logging.getLogger("ultralytics").setLevel(logging.ERROR)


/tmp/ipykernel_27437/2828595318.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# Block 2: Workspace and lightweight video validation for direct inference/tracking

def initialize_workspace():
    base = Path.cwd().absolute()
    paths = {
        "videos": base / "raw_data" / "videos",
        "tracks": base / "Runs" / "trajectories",
        "renders": base / "Runs" / "annotated_videos",
        "models": base / "models",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def open_video_capture(video_path):
    video_path = Path(video_path)

    backends = [cv2.CAP_MSMF, cv2.CAP_DSHOW, cv2.CAP_ANY] if os.name == "nt" else [cv2.CAP_ANY]

    for api in backends:
        cap = cv2.VideoCapture(str(video_path), api)
        if cap.isOpened():
            ret, _ = cap.read()
            if ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                return cap
            cap.release()

    return None


def get_video_metadata(video_path):
    cap = open_video_capture(video_path)
    if cap is None:
        return None

    meta = {
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "total": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return meta


def validate_video_input(video_path):
    video_path = Path(video_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap = open_video_capture(video_path)
    if cap is None:
        return False

    cap.release()
    return True

In [3]:
# Block 3: Strict 7-Class Mapping (Rulebook Compliant)
def get_grouped_class(source_label, car_group):
    # Rulebook Mapping: Forces every AI detection into one of the 7 approved classes
    mapping = {
        "CAR_GROUP": "Car", "Car": "Car", "Hatchback": "Car", "Sedan": "Car", "SUV": "Car", "MUV": "Car", "Van": "Car",
        "Truck": "HCV", "HCV": "HCV", "Tractor": "HCV",
        "Auto Rickshaw": "Three-wheeler", "Rickshaw": "Three-wheeler", "Three-wheeler": "Three-wheeler",
        "Motorcycle": "Two-wheeler", "Bicycle": "Two-wheeler", "Two-wheeler": "Two-wheeler",
        "Mini Bus": "Bus", "Bus": "Bus",
        "Tempo": "LCV", "LCV": "LCV",
        "person": "Pedestrian", "Pedestrian": "Pedestrian"
    }
    # Safe fallback to "Car" to prevent invalid labels in XML
    return mapping.get(source_label, "Car")

def get_ultra_tracker_yaml(mode):
    if mode.endswith(".yaml"): return mode
    if mode == "botsort": return "botsort.yaml"
    if mode == "bytetrack": return "bytetrack.yaml"
    raise ValueError(f"Unsupported tracker_mode: {mode}")


In [4]:
# Block 4: Optimized Trajectory Extraction (High-Res + Filtered)

def extract_trajectories(model, v_path, c_path, out_path, meta, cfg):
    cap = open_video_capture(v_path)
    if cap is None: raise ValueError(f"Could not open video: {v_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, cfg["start_frame"])
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), meta["fps"], (meta["width"], meta["height"]))
    data_log = []
    ultra_yaml = get_ultra_tracker_yaml(cfg["tracker_mode"])
    
    for frame_idx in tqdm(range(cfg["start_frame"], cfg["end_frame"]), desc="Tracking"):
        ret, frame = cap.read()
        if not ret: break
        
        # Inference with High-Res (imgsz=1080) for better vehicle separation
        results = model.track(frame, persist=True, tracker=ultra_yaml, conf=cfg["conf"], iou=cfg["iou"], imgsz=1080, verbose=False)
        
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            clss = results[0].boxes.cls.cpu().numpy().astype(int)
            confs = results[0].boxes.conf.cpu().numpy()
            
            for box, tid, c_idx, score in zip(boxes, ids, clss, confs):
                src_label = model.names[c_idx]
                grp_label = get_grouped_class(src_label, cfg["car_group"])
                data_log.append({
                    "frame": frame_idx, "track_id": tid, "grouped_class": grp_label,
                    "source_class": src_label, "confidence": float(score), "bbox": [round(float(v), 2) for v in box]
                })
                # Visual Check: Everything is Green for speed
                x1, y1, x2, y2 = [int(v) for v in box]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"ID:{tid} {grp_label}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        writer.write(frame)
        
    cap.release(); writer.release()
    if data_log:
        pd.DataFrame(data_log).to_csv(c_path, index=False)
    else:
        with open(c_path, "w") as f: f.write("frame,track_id,grouped_class,source_class,confidence,bbox")


In [5]:
# Block 5: Annotated video features
def build_display_color_map(model_names, car_group):
    """
    Build a stable, explicit colour map.
    CAR_GROUP is forced to green.
    Common non-group labels get fixed colours.
    Any remaining labels get colours from a fallback palette that does not use green.
    """
    if isinstance(model_names, dict):
        labels = list(model_names.values())
    else:
        labels = list(model_names)

    # Keep original order but remove duplicates
    ordered_labels = []
    for lab in labels:
        lab = str(lab)
        if lab not in ordered_labels:
            ordered_labels.append(lab)

    color_map = {
        "CAR_GROUP": (0, 255, 0),   # green, fixed
        "LCV": (255, 0, 0),         # blue
        "Bus": (0, 165, 255),       # orange
        "Mini Bus": (255, 0, 255),  # magenta
        "Truck": (0, 0, 255),       # red
        "Auto Rickshaw": (255, 255, 0),
        "Rickshaw": (0, 255, 255),
        "Bicycle": (255, 128, 0),
        "Motorcycle": (128, 0, 255),
        "Tempo": (128, 128, 0),
    }

    fallback_palette = [
        (255, 128, 0),
        (128, 0, 255),
        (0, 255, 255),
        (255, 0, 255),
        (0, 0, 255),
        (0, 165, 255),
        (255, 255, 0),
        (128, 128, 0),
        (0, 128, 255),
        (255, 64, 64),
        (64, 255, 64),
        (64, 64, 255),
        (200, 100, 0),
        (100, 0, 200),
    ]

    used = set(color_map.values())
    fallback_idx = 0

    for label in ordered_labels:
        if label in car_group:
            color_map[label] = (0, 255, 0)
            continue

        if label in color_map:
            continue

        # Pick the next unused fallback colour
        while fallback_idx < len(fallback_palette) and fallback_palette[fallback_idx] in used:
            fallback_idx += 1

        if fallback_idx < len(fallback_palette):
            color_map[label] = fallback_palette[fallback_idx]
            used.add(fallback_palette[fallback_idx])
            fallback_idx += 1
        else:
            # Last-resort distinct colour, still not green
            color_map[label] = (180, 180, 60)

    return color_map




In [6]:
def build_cvat_video_xml(df, xml_out_path, task_name, task_size, video_width, video_height):
    import xml.etree.ElementTree as ET
    from tqdm import tqdm
    from collections import Counter
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"
    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    ET.SubElement(task, "name").text = str(task_name)
    ET.SubElement(task, "size").text = str(int(task_size))
    ET.SubElement(task, "mode").text = "interpolation"
    labels_el = ET.SubElement(task, "labels")
    for label_name in sorted(df["grouped_class"].unique()):
        l = ET.SubElement(labels_el, "label")
        ET.SubElement(l, "name").text = label_name
        ET.SubElement(l, "type").text = "bbox"
    osize = ET.SubElement(task, "original_size")
    ET.SubElement(osize, "width").text = str(int(video_width))
    ET.SubElement(osize, "height").text = str(int(video_height))
    max_valid_frame = int(task_size) - 1
    
    for tid, track_df in tqdm(df.groupby("track_id"), desc="Task 5: CVAT XML"):
        track_label = Counter(track_df["grouped_class"].tolist()).most_common(1)[0][0]
        track_el = ET.SubElement(root, "track", id=str(int(tid)), label=track_label)
        track_df = track_df.sort_values("frame")
        
        # FIXED: Only mark every 10th frame as a Keyframe to allow manual interpolation
        for i, row in enumerate(track_df.itertuples()):
            if int(row.frame) <= max_valid_frame:
                is_key = "1" if (i == 0 or i == len(track_df)-1 or i % 10 == 0) else "0"
                ET.SubElement(track_el, "box", frame=str(int(row.frame)), xtl=f"{row.xtl:.2f}", ytl=f"{row.ytl:.2f}", xbr=f"{row.xbr:.2f}", ybr=f"{row.ybr:.2f}", outside="0", keyframe=is_key)
        
        last_f = int(track_df["frame"].max())
        if last_f + 1 <= max_valid_frame:
            ET.SubElement(track_el, "box", frame=str(last_f + 1), xtl=f"{track_df.iloc[-1].xtl:.2f}", ytl=f"{track_df.iloc[-1].ytl:.2f}", xbr=f"{track_df.iloc[-1].xbr:.2f}", ybr=f"{track_df.iloc[-1].ybr:.2f}", outside="1", keyframe="1")

    ET.indent(root, space="  ", level=0)
    tree = ET.ElementTree(root)
    tree.write(xml_out_path, encoding="utf-8", xml_declaration=True)


In [7]:
# Block 7: Validation
def validate_cvat_xml(xml_path):
    """
    Load a CVAT Video 1.1 XML file and perform basic structural checks.
    This does not import into CVAT; it only verifies that the file is well-formed
    and contains the expected annotation elements.
    """
    xml_path = Path(xml_path)

    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found: {xml_path}")

    tree = ET.parse(xml_path)
    root = tree.getroot()

    if root.tag != "annotations":
        raise ValueError(f"Unexpected root tag: {root.tag}. Expected 'annotations'.")

    version_el = root.find("version")
    if version_el is None:
        raise ValueError("Missing <version> element in XML.")
    print(f"XML version: {version_el.text}")

    meta = root.find("meta")
    if meta is None:
        raise ValueError("Missing <meta> section in XML.")

    task = meta.find("task")
    if task is None:
        raise ValueError("Missing <task> section inside <meta>.")

    name_el = task.find("name")
    mode_el = task.find("mode")
    size_el = task.find("size")

    print(f"Task name: {name_el.text if name_el is not None else 'N/A'}")
    print(f"Task mode: {mode_el.text if mode_el is not None else 'N/A'}")
    print(f"Task size: {size_el.text if size_el is not None else 'N/A'}")

    labels = task.find("labels")
    if labels is None:
        raise ValueError("Missing <labels> section inside <task>.")

    label_names = []
    for label in labels.findall("label"):
        name = label.find("name")
        if name is not None and name.text:
            label_names.append(name.text)

    if not label_names:
        raise ValueError("No labels found in XML.")

    print(f"Labels found ({len(label_names)}): {label_names}")

    tracks = root.findall("track")
    if not tracks:
        raise ValueError("No <track> elements found in XML.")

    print(f"Tracks found: {len(tracks)}")

    print("\nSample track inspection:")
    for track in tracks[:3]:
        track_id = track.attrib.get("id", "N/A")
        track_label = track.attrib.get("label", "N/A")
        boxes = track.findall("box")
        print(f"  Track id={track_id}, label={track_label}, boxes={len(boxes)}")

        for box in boxes[:2]:
            frame = box.attrib.get("frame", "N/A")
            xtl = box.attrib.get("xtl", "N/A")
            ytl = box.attrib.get("ytl", "N/A")
            xbr = box.attrib.get("xbr", "N/A")
            ybr = box.attrib.get("ybr", "N/A")
            outside = box.attrib.get("outside", "N/A")
            keyframe = box.attrib.get("keyframe", "N/A")
            print(
                f"    box frame={frame}, xtl={xtl}, ytl={ytl}, "
                f"xbr={xbr}, ybr={ybr}, outside={outside}, keyframe={keyframe}"
            )

    print(f"\nValidation passed: {xml_path}")

In [8]:
# Block 8: Tracking
def print_track_diagnostics(csv_path):
    df = pd.read_csv(csv_path)

    track_sizes = df.groupby("track_id").size().sort_values()

    print("Total rows:", len(df))
    print("Unique track_ids:", df["track_id"].nunique())
    print("\nTrack length summary:")
    print(track_sizes.describe())

    print("\nSingle-frame tracks:", int((track_sizes == 1).sum()))
    print("Tracks with 2-3 frames:", int(((track_sizes >= 2) & (track_sizes <= 3)).sum()))
    print("Tracks with >50 frames:", int((track_sizes > 50).sum()))

    duplicate_pairs = df.duplicated(["frame", "track_id"], keep=False).sum()
    print("\nDuplicate (frame, track_id) rows:", int(duplicate_pairs))

    print("\nSmallest tracks:")
    print(track_sizes.head(20))

    print("\nLargest tracks:")
    print(track_sizes.tail(20))

In [9]:
# Block 9: Execution (YOLOv11 + GPU)
import torch
import ast
PROJ_PATHS = initialize_workspace()
CONFIG = {
    "start_frame": 0, "end_frame": 23261, "conf": 0.45, "iou": 0.1, 
    "tracker_mode": "dense_traffic_tracker.yaml", "car_group": {"Car"}
}
VIDEO_IN = PROJ_PATHS["videos"] / "76_RC_PTZ 1_20260427_210000_20260427_211500_4.mp4"
CSV_OUT = PROJ_PATHS["tracks"] / "trajectories24.csv"
XML_OUT = PROJ_PATHS["tracks"] / "cvat_annotations24.xml"
VIDEO_OUT = PROJ_PATHS["renders"] / "annotated_junction24.mp4"
MODEL_PATH = "models/UVH-26-MV-YOLOv11-S.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
traffic_model = YOLO(MODEL_PATH).to(device)
video_meta = get_video_metadata(VIDEO_IN)

# Run optimized pipeline
extract_trajectories(traffic_model, VIDEO_IN, CSV_OUT, VIDEO_OUT, video_meta, CONFIG)

# Clean data and Export to CVAT XML
df = pd.read_csv(CSV_OUT)
if not df.empty:
    # --- Fix: Expand bbox coordinates ---
    def parse_bb(x):
        try: return ast.literal_eval(x) if isinstance(x, str) else x
        except: return [0,0,0,0]
    
    coords = pd.DataFrame(df["bbox"].apply(parse_bb).tolist(), columns=["xtl", "ytl", "xbr", "ybr"], index=df.index)
    df = pd.concat([df, coords], axis=1)
    
    # Filter short tracks
    counts = df["track_id"].value_counts()
    df = df[df["track_id"].isin(counts[counts >= 15].index)]
    
    build_cvat_video_xml(df, XML_OUT, VIDEO_IN.stem, video_meta["total"], video_meta["width"], video_meta["height"])
    print(f"--- Success (Using {device}) ---")
else:
    print("--- Pipeline Finished with NO Detections ---")


Task 5: CVAT XML: 100%|██████████| 1679/1679 [00:06<00:00, 256.21it/s]


--- Success (Using cuda) ---
